In [2]:
from datasets import load_dataset
import pandas as pd

ds = load_dataset("kurry/sp500_earnings_transcripts", split="train")
print(ds)                      
print(ds.column_names)

Dataset({
    features: ['symbol', 'quarter', 'year', 'date', 'content', 'structured_content', 'company_name', 'company_id'],
    num_rows: 33362
})
['symbol', 'quarter', 'year', 'date', 'content', 'structured_content', 'company_name', 'company_id']


In [3]:
# Light table without the huge text columns
meta = ds.remove_columns(["content", "structured_content"]).to_pandas()
meta["date"] = pd.to_datetime(meta["date"])

# 1. Empty transcripts
n_chars = ds.map(lambda x: {"n": len(x["content"] or "")},
                 remove_columns=ds.column_names)["n"]
meta["n_chars"] = n_chars
print("empty:", (meta.n_chars == 0).sum(), " very short (<5000):", (meta.n_chars < 5000).sum())

# 2. Does `date` contain a real time of day, or just midnight?
print(meta["date"].dt.time.value_counts().head())

# 3. Duplicates
print("dupes:", meta.duplicated(["symbol", "year", "quarter"]).sum())

# 4. Calls per company
print(meta.groupby("symbol").size().describe())

Map:   0%|          | 0/33362 [00:00<?, ? examples/s]

empty: 1  very short (<5000): 136
date
17:00:00    4709
08:30:00    4589
10:00:00    4349
09:00:00    3933
11:00:00    3290
Name: count, dtype: int64
dupes: 0
count    685.000000
mean      48.703650
std       23.174801
min        1.000000
25%       27.000000
50%       55.000000
75%       70.000000
max       79.000000
dtype: float64


In [4]:
import json, pathlib
raw = pathlib.Path("data/raw"); raw.mkdir(parents=True, exist_ok=True)

def save_call(symbol, year, quarter):
    rows = ds.filter(lambda x: x["symbol"] == symbol and x["year"] == year
                     and x["quarter"] == quarter and len(x["content"] or "") > 0)
    if len(rows) == 0:
        print("not found:", symbol, year, quarter); return
    r = rows[0]
    stem = f"{symbol}_{year}Q{quarter}"
    (raw / f"{stem}.txt").write_text(r["content"], encoding="utf-8")        # untouched raw text
    (raw / f"{stem}.json").write_text(json.dumps(r["structured_content"]))  # speaker-split version
    print(stem, r["date"], len(r["content"]))

save_call("AAPL", 2022, 2)
save_call("JPM", 2019, 3)
save_call("XOM", 2016, 1)

Filter:   0%|          | 0/33362 [00:00<?, ? examples/s]

AAPL_2022Q2 2022-04-28 17:00:00 47260


Filter:   0%|          | 0/33362 [00:00<?, ? examples/s]

JPM_2019Q3 2019-10-15 08:30:00 45226


Filter:   0%|          | 0/33362 [00:00<?, ? examples/s]

XOM_2016Q1 2016-04-29 09:30:00 61305
